Partie 1 — Installation des dépendances et importation des bibliothèques
cette étape prépare l'environnement de travail. Nous installons les bibliothèques nécessaires à la manipulation des données, au traitement audio et à la visualisation statistique.

In [ ]:

# Installation des bibliothèques nécessaires au projet.
# - datasets : chargement et manipulation des datasets Hugging Face.
# - soundfile et librosa : lecture et traitement des fichiers audio.
# - pandas : manipulation des données sous forme de tableaux.
# - matplotlib : création de graphiques pour visualiser les résultats.
!pip install -q datasets soundfile librosa pandas matplotlib

# Importation des bibliothèques utilisées dans le notebook.
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
import numpy as np

# Configuration de l'affichage pandas pour visualiser les textes
# sans les tronquer après 120 caractères.
pd.set_option("display.max_colwidth", 120)


Partie 2 — Connexion à Google Drive
Google Colab utilise un environnement temporaire. Pour éviter de perdre les datasets et les fichiers générés après la fermeture de la session, nous utilisons Google Drive comme espace de stockage persistant.

In [ ]:
## 1bis. Connexion a Google Drive (stockage persistant)

# Importation de la fonction permettant de connecter Google Drive à Colab.
from google.colab import drive
import os

# Montage de Google Drive dans l'environnement Colab.
# Une fenêtre d'autorisation Google apparaît lors de la première exécution.
drive.mount('/content/drive')

# Définition du dossier principal du projet sur Google Drive.
# Tous les fichiers du projet seront organisés à partir de ce dossier.
PROJECT_DIR = "/content/drive/MyDrive/darija-transcription-translation"

# Création des sous-dossiers nécessaires au projet.
# - data/raw : stockage des datasets originaux.
# - data/processed : stockage des données préparées et filtrées.
# - models : stockage des modèles et de leurs checkpoints.
# exist_ok=True évite une erreur si les dossiers existent déjà.
for sub in ["data/raw", "data/processed", "models"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

# Affichage du chemin principal et du contenu du dossier.
print("Projet sur Drive pret a :", PROJECT_DIR)
print(os.listdir(PROJECT_DIR))


Partie 3 — Authentification Hugging Face
 certains datasets sont soumis à des conditions d'accès. L'authentification permet de vérifier que l'utilisateur dispose des autorisations nécessaires pour télécharger ces données.

In [ ]:
# Importation de la fonction login pour s'authentifier sur Hugging Face.
from huggingface_hub import login

try:
    # Sur Google Colab, récupération du token depuis les Secrets.
    # Le token permet d'accéder aux datasets protégés.
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')

except (ImportError, Exception):
    # En environnement local, récupération du token depuis
    # les variables d'environnement du système.
    # Il est préférable de ne pas écrire directement le token
    # dans le notebook pour des raisons de sécurité.
    import os
    hf_token = os.environ.get('HF_TOKEN')

# Vérification de la disponibilité du token.
if hf_token:

    # Authentification auprès de la plateforme Hugging Face.
    login(token=hf_token)

    # Confirmation de la réussite de l'authentification.
    print("Authentification HuggingFace reussie.")

else:

    # Message d'avertissement si aucun token n'est disponible.
    print("ATTENTION: HF_TOKEN non trouve. Voir les instructions ci-dessus.")


Partie 4 — Exploration du dataset ASR
Objectif : examiner le dataset audio DODa afin de comprendre sa structure, ses colonnes, ses formats audio et les informations textuelles associées.

4.1 Chargement du dataset
Le code vérifie d'abord si le dataset existe déjà sur Google Drive. S'il existe, il est chargé directement depuis le stockage local. Sinon, il est téléchargé depuis Hugging Face puis sauvegardé sur Drive.

Cette méthode évite de télécharger plusieurs fois les mêmes données.

In [ ]:
## 2. Dataset ASR : atlasia/DODa-audio-dataset

# Importation des fonctions nécessaires au chargement des datasets.
import os
from datasets import load_dataset, load_from_disk

# Définition du chemin de stockage du dataset audio sur Google Drive.
asr_cache_path = os.path.join(PROJECT_DIR, "data/raw/doda_audio_dataset")

# Vérification de l'existence du dataset dans le stockage local.
if os.path.exists(asr_cache_path):

    # Si le dataset existe déjà, on le charge directement depuis Drive.
    # Cela évite un nouveau téléchargement.
    print("Chargement depuis le cache Drive (pas de re-telechargement)...")
    asr_ds = load_from_disk(asr_cache_path)

else:

    # Si le dataset n'existe pas, on le télécharge depuis Hugging Face.
    print("Telechargement depuis le Hub HuggingFace...")
    asr_ds = load_dataset("atlasia/DODa-audio-dataset")

    # Sauvegarde du dataset sur Google Drive pour les prochaines exécutions.
    print("Sauvegarde sur Drive pour la prochaine fois...")
    asr_ds.save_to_disk(asr_cache_path)

# Affichage de la structure générale du dataset.
# Cette sortie permet de voir les splits et les caractéristiques des données.
print(asr_ds)


4.2 Inspection de la structure et du premier exemple
Nous examinons les colonnes disponibles ainsi qu'un premier exemple pour comprendre les informations contenues dans le dataset.

In [ ]:
# Sélection du premier split disponible dans le dataset.
split_name = list(asr_ds.keys())[0]

# Affichage des noms des colonnes disponibles.
# Ces colonnes indiquent les informations associées à chaque exemple.
print("Colonnes:", asr_ds[split_name].column_names)
print()

# Récupération du premier exemple du dataset.
example = asr_ds[split_name][0]


def get_audio_array_and_sr(audio_value, debug=False):
    """
    Cette fonction permet de récupérer les données audio
    et leur fréquence d'échantillonnage.

    Elle prend en charge plusieurs formats possibles selon
    la version de la bibliothèque datasets :

    - Format dict : {"array": ..., "sampling_rate": ...}
    - Format AudioDecoder : utilisé dans les versions récentes
      de datasets avec le backend torchcodec.
    """

    # Premier cas : les données audio sont représentées sous forme de dictionnaire.
    if isinstance(audio_value, dict):

        # Extraction du signal audio et de sa fréquence d'échantillonnage.
        arr, sr = audio_value["array"], audio_value["sampling_rate"]

        # Affichage des informations techniques en mode diagnostic.
        if debug:
            print(f"[debug] format dict -> arr.shape={getattr(arr, 'shape', len(arr))}, sr={sr}")

        # Retour du signal audio et de sa fréquence.
        return arr, sr

    # Deuxième cas : les données sont représentées par un objet AudioDecoder.
    if debug:
        print(f"[debug] type audio_value: {type(audio_value)}")

        # Affichage des attributs disponibles pour comprendre la structure de l'objet.
        print(f"[debug] attributs disponibles: {[a for a in dir(audio_value) if not a.startswith('_')]}")

    # Récupération de tous les échantillons audio.
    samples = audio_value.get_all_samples()

    # Affichage des informations techniques pour faciliter le diagnostic.
    if debug:
        print(f"[debug] type samples: {type(samples)}")
        print(f"[debug] attributs samples: {[a for a in dir(samples) if not a.startswith('_')]}")
        print(f"[debug] samples.data.shape (brut): {samples.data.shape}")
        print(f"[debug] samples.sample_rate: {samples.sample_rate}")

    # Conversion des données audio en tableau NumPy.
    array = samples.data.numpy()

    # Vérification du nombre de dimensions du signal audio.
    # Si le signal contient plusieurs canaux (stéréo),
    # on calcule la moyenne pour obtenir un signal mono.
    # Cela facilite le traitement audio par les modèles ASR.
    if array.ndim == 2:
        array = array.mean(axis=0)

    # Récupération de la fréquence d'échantillonnage.
    sr = samples.sample_rate

    # Affichage du résultat final en mode diagnostic.
    if debug:
        print(f"[debug] array.shape (final): {array.shape}, sr final: {sr}")

    # Retour du signal audio et de sa fréquence d'échantillonnage.
    return array, sr


# Premier appel de la fonction en mode diagnostic.
# L'objectif est de vérifier le format et les dimensions du signal audio.
print("=== DIAGNOSTIC sur le premier exemple ===")
arr_debug, sr_debug = get_audio_array_and_sr(example["audio"], debug=True)

# Calcul de la durée du signal :
# durée = nombre d'échantillons / fréquence d'échantillonnage.
print(f"=== Resultat: shape={arr_debug.shape}, sr={sr_debug}, duree={len(arr_debug)/sr_debug:.3f}s ===")
print()

# Affichage des autres informations du premier exemple.
# La colonne audio est exclue car elle contient les données du signal.
for k, v in example.items():
    if k != "audio":
        print(f"{k}: {v}")


4.3 Écoute et observation d'un exemple audio
Cette étape permet de vérifier concrètement le contenu du dataset en écoutant un enregistrement et en comparant le signal audio avec ses différentes transcriptions.

In [ ]:
# Importation du module permettant de lire un fichier audio dans le notebook.
import IPython.display as ipd

# Extraction du signal audio et de sa fréquence d'échantillonnage.
audio_array, sr = get_audio_array_and_sr(example["audio"])

# Affichage des différentes représentations textuelles de l'audio.
# Cela permet de comparer les transcriptions selon les écritures utilisées.
print("Texte (Latin)        :", example.get("darija_Latn", "N/A"))
print("Texte (Arabe, new)   :", example.get("darija_Arab_new", "N/A"))
print("Texte (Arabe, old)   :", example.get("darija_Arab_old", "N/A"))
print("Traduction EN        :", example.get("english", "N/A"))

# Lecture de l'enregistrement audio directement dans le notebook.
# Le paramètre rate correspond à la fréquence d'échantillonnage du signal.
ipd.Audio(audio_array, rate=sr)


Interprétation :

Cette opération permet de vérifier la correspondance entre la parole prononcée et les transcriptions disponibles. Elle donne également une première idée de la diversité des écritures utilisées pour représenter la Darija.

4.4 Analyse statistique des durées audio
L'objectif est d'étudier la durée des enregistrements afin de mieux connaître les caractéristiques du dataset.

Pour limiter le temps d'exécution, nous utilisons un échantillon de 500 exemples au maximum.

In [ ]:
# Définition du nombre d'exemples à analyser.
# On utilise au maximum 500 exemples pour réduire le temps de calcul.
n_sample = min(500, len(asr_ds[split_name]))

# Liste destinée à stocker les durées des enregistrements audio.
durations = []

# Parcours des exemples sélectionnés.
for i in range(n_sample):

    # Extraction du signal audio et de sa fréquence d'échantillonnage.
    arr, sr_i = get_audio_array_and_sr(asr_ds[split_name][i]["audio"])

    # Calcul de la durée de l'enregistrement en secondes.
    durations.append(len(arr) / sr_i)

# Affichage du nombre total d'exemples dans le dataset.
print("Nombre total d'exemples:", len(asr_ds[split_name]))

# Calcul et affichage des durées minimale et maximale.
print(f"Duree min: {min(durations):.3f}s, Duree max: {max(durations):.3f}s")

# Calcul de la durée moyenne sur l'échantillon analysé.
print(f"Duree moyenne (sur {n_sample} ech.): {np.mean(durations):.2f}s")

# Estimation de la durée totale du dataset en heures.
# Cette estimation suppose que l'échantillon est représentatif du dataset complet.
print(f"Duree totale estimee (extrapolee): {np.mean(durations) * len(asr_ds[split_name]) / 3600:.2f}h")

# Vérification d'un éventuel problème lors de l'extraction des durées.
if np.mean(durations) == 0:

    # Affichage d'un avertissement si la durée moyenne est nulle.
    print()
    print("ATTENTION: duree moyenne = 0, quelque chose ne va pas dans l'extraction.")

    # Indication de la fonction à relancer pour examiner le problème.
    print("Relancer la cellule precedente avec debug=True pour investiguer:")
    print('  get_audio_array_and_sr(asr_ds[split_name][0]["audio"], debug=True)')

# Création d'une figure pour représenter la distribution des durées audio.
plt.figure(figsize=(8,4))

# Création de l'histogramme des durées.
# bins=30 divise les durées en 30 intervalles.
plt.hist(durations, bins=30)

# Ajout du titre et des noms des axes.
plt.title("Distribution des durees audio (echantillon)")
plt.xlabel("Duree (secondes)")
plt.ylabel("Nombre d'exemples")

# Affichage du graphique.
plt.show()


Interprétation attendue :

L'histogramme permet d'observer la répartition des durées des enregistrements audio. Il aide à identifier les enregistrements courts, les enregistrements longs et les éventuelles valeurs extrêmes.

La durée moyenne donne une indication sur la quantité de données audio à traiter lors de l'entraînement du modèle Whisper.

Partie 5 — Exploration du dataset de traduction
Objectif : explorer le dataset MBZUAI-Paris/Darija-SFT-Mixture, identifier les différentes directions de traduction et extraire uniquement les exemples correspondant à notre besoin : Darija vers Français.

5.1 Chargement du dataset de traduction
Comme pour le dataset audio, nous vérifions d'abord si les données existent déjà sur Google Drive. Si elles sont disponibles, nous les chargeons depuis le stockage local. Sinon, nous les téléchargeons depuis Hugging Face.

In [ ]:
## 3. Dataset traduction : MBZUAI-Paris/Darija-SFT-Mixture

# Définition du chemin de stockage du dataset de traduction.
translation_cache_path = os.path.join(PROJECT_DIR, "data/raw/darija_sft_mixture_full")

# Vérification de la présence du dataset sur Google Drive.
if os.path.exists(translation_cache_path):

    # Chargement depuis le stockage local si le dataset existe déjà.
    print("Chargement depuis le cache Drive (pas de re-telechargement)...")
    translation_ds = load_from_disk(translation_cache_path)

else:

    # Téléchargement du dataset depuis Hugging Face.
    print("Telechargement depuis le Hub HuggingFace...")
    translation_ds = load_dataset("MBZUAI-Paris/Darija-SFT-Mixture")

    # Sauvegarde du dataset complet pour éviter de le télécharger à nouveau.
    print("Sauvegarde sur Drive pour la prochaine fois...")
    translation_ds.save_to_disk(translation_cache_path)

# Affichage de la structure générale du dataset.
print(translation_ds)

# Sélection du premier split disponible.
split_name_t = list(translation_ds.keys())[0]

# Affichage des colonnes disponibles dans le dataset.
print("Colonnes:", translation_ds[split_name_t].column_names)
print()

# Affichage du premier exemple pour comprendre son organisation.
print(translation_ds[split_name_t][0])
print()

# Conversion du dataset en DataFrame pandas.
# Cette représentation facilite les opérations de filtrage,
# d'analyse statistique et d'extraction des données textuelles.
df_t = translation_ds[split_name_t].to_pandas()

# Vérification de la création du DataFrame et de sa taille.
print(f"df_t cree avec succes : {len(df_t)} lignes")


5.2 Identification des directions de traduction
Le dataset contient plusieurs directions de traduction. Nous utilisons la colonne direction pour identifier les différentes combinaisons de langues disponibles.

Par exemple :
- dr_fr : Darija vers Français.
- fr_dr : Français vers Darija.
- dr_en : Darija vers Anglais.

In [ ]:
# La colonne direction indique le sens de traduction de chaque exemple.
# value_counts() permet de compter le nombre d'occurrences de chaque direction.

print("Valeurs uniques de la colonne direction:")
print(df_t["direction"].value_counts())

Interprétation :

Cette analyse permet de connaître la répartition des exemples selon les directions de traduction.

Elle est importante pour identifier précisément le sous-ensemble correspondant à notre objectif, sans mélanger les traductions dans des sens différents.

5.3 Filtrage des paires Darija → Français
Nous appliquons un filtrage strict à partir de la colonne direction.

Ce choix permet d'éviter une sélection approximative basée sur la recherche de mots dans les textes, qui pourrait introduire des exemples non pertinents.

In [ ]:
# Filtrage strict sur la colonne "direction".
# Nous conservons uniquement les exemples dont la direction est dr_fr.
# Cette méthode est plus fiable qu'une recherche de mots dans le contenu
# des textes, qui pourrait sélectionner des exemples incorrects.

# Définition du code correspondant à la direction Darija vers Français.
# Cette valeur doit correspondre aux résultats affichés dans la cellule précédente.
FRENCH_DIRECTION = "dr_fr"  # <-- ajuster selon la sortie de la cellule precedente si besoin

# Sélection des exemples correspondant uniquement à la direction choisie.
# copy() permet de créer une copie indépendante du sous-ensemble filtré.
df_fr = df_t[df_t["direction"] == FRENCH_DIRECTION].copy()

# Affichage du nombre de paires Darija-Français obtenues.
print(f"Nombre de paires darija -> francais (direction == '{FRENCH_DIRECTION}'): {len(df_fr)}")


5.4 Extraction des textes source et cible
Les conversations du dataset sont organisées sous forme de messages associés à des rôles : user et assistant.

Nous devons donc extraire :
- Le message user, qui contient le texte source et éventuellement une instruction.
- Le message assistant, qui contient la réponse ou la traduction.
La fonction suivante réalise cette extraction.

In [ ]:
def extract_user_assistant(messages):
    """
    Extrait les contenus des messages user et assistant
    depuis la colonne messages.

    Le message user contient l'instruction et la phrase source.
    Le message assistant contient la traduction produite.
    """

    # Initialisation des deux variables.
    user_text, assistant_text = None, None

    # Parcours de la liste des messages.
    for m in messages:

        # Récupération du contenu du message utilisateur.
        if m.get("role") == "user":
            user_text = m.get("content")

        # Récupération du contenu de la réponse de l'assistant.
        elif m.get("role") == "assistant":
            assistant_text = m.get("content")

    # Retour des deux contenus extraits.
    return user_text, assistant_text


# Application de la fonction à chaque ligne de la colonne messages.
# Les résultats sont placés dans deux nouvelles colonnes :
# source_text : contenu du message utilisateur.
# target_text : contenu de la réponse de l'assistant.
df_fr[["source_text", "target_text"]] = df_fr["messages"].apply(
    lambda msgs: pd.Series(extract_user_assistant(msgs))
)

# Affichage des dix premières paires extraites.
# Cela permet de vérifier visuellement la structure des données.
df_fr[["direction", "source_text", "target_text"]].head(10)


Interprétation :

Cette étape transforme les conversations structurées en paires textuelles exploitables pour la traduction automatique.

La colonne source_text représente le texte d'entrée et la colonne target_text représente le texte de sortie attendu.

Ces deux colonnes constituent les éléments essentiels pour préparer les données d'entraînement d'un modèle de traduction.

5.5 Sauvegarde des données filtrées
Après l'extraction, nous sauvegardons les paires Darija-Français dans un fichier CSV.

Nous conservons uniquement les colonnes utiles pour les prochaines étapes du projet afin d'obtenir un fichier plus simple à exploiter.

In [ ]:
# Sauvegarde du sous-ensemble filtré et extrait pour la phase de prétraitement.

# Importation du module os pour gérer les chemins de fichiers.
import os

# Définition du chemin de sortie du fichier CSV.
# Le fichier sera enregistré dans data/processed/.
output_path = os.path.join(PROJECT_DIR, "data/processed/translation_pairs_fr_raw.csv")

# Sélection des colonnes nécessaires pour la suite du projet.
# On exclut la colonne messages qui contient des structures imbriquées.
cols_to_keep = ["dataset", "id", "direction", "source_text", "target_text"]

# Exportation des données sélectionnées au format CSV.
# index=False évite d'ajouter une colonne contenant les indices pandas.
df_fr[cols_to_keep].to_csv(output_path, index=False)

# Affichage du chemin de sauvegarde.
print(f"Sauvegarde -> {output_path}")

# Affichage du nombre de paires sauvegardées.
print(f"Nombre de paires sauvegardees: {len(df_fr)}")


Interprétation :

Le fichier translation_pairs_fr_raw.csv contient les paires de traduction extraites du dataset original.

Il représente une première version des données préparées. Le suffixe raw indique que ces données n'ont pas encore subi toutes les opérations de nettoyage et de normalisation.

Ce fichier sera utilisé dans la phase suivante pour le prétraitement des données de traduction.

Partie 6 — Prochaines étapes

- Passer a `scripts/asr_data.py` et `scripts/translation_data.py` (Phase 2)